# 03 — Retrieval with LlamaIndex
This notebook demonstrates **retrieval** using a LlamaIndex `VectorStoreIndex`.

> **Run cells top to bottom.** All setup (imports, Settings, document loading) must be executed before retrieval.

In [1]:
# Cell 1 — Load environment variables
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# Cell 2 — Configure LLM and Embedding Settings
import os
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.openai import OpenAI

Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
)

Settings.llm = OpenAI(
    model="gpt-4o",
    api_key=os.getenv("OPENAI_API_KEY"),
)

print("Settings configured successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Settings configured successfully.


In [3]:
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import SimpleDirectoryReader
docs = SimpleDirectoryReader(
    input_files=[
        r"D:\vector_databse\projeect for rag pipeline\dataset\hr_policy_manual.pdf",
        r"D:\vector_databse\projeect for rag pipeline\dataset\novacrm_product_manual.pdf",
        r"D:\vector_databse\projeect for rag pipeline\dataset\annual_financial_report_2024.pdf",
    ]
).load_data()

nodes = SentenceSplitter(chunk_size=512, chunk_overlap=64).get_nodes_from_documents(docs)

In [4]:
import chromadb
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore

chroma_client = chromadb.PersistentClient(path="./nb03_chroma")
col = chroma_client.get_or_create_collection("doc_v1")
vs = ChromaVectorStore(chroma_collection=col)
sc = StorageContext.from_defaults(vector_store=vs)

# define your nodes first, e.g.:
# from llama_index.core.node_parser import SentenceSplitter
# nodes = SentenceSplitter().get_nodes_from_documents(documents)

index = VectorStoreIndex(nodes=nodes, storage_context=sc)

In [5]:
vs

ChromaVectorStore(stores_text=True, is_embedding_query=True, flat_metadata=True, collection_name=None, host=None, port=None, ssl=False, headers=None, persist_dir=None, collection_kwargs={})

In [6]:
index = VectorStoreIndex.from_vector_store(vs,storage_context= sc)

In [7]:
from llama_index.core.base.embeddings.base import similarity
query = "What is the WFH policy?"

retriever = index.as_retriever(similarity_top_k=3)
retrieved_nodes = retriever.retrieve(query)

In [8]:
retrieved_nodes

[NodeWithScore(node=TextNode(id_='1c570dfe-1472-4d5e-afbe-56a8870eac28', embedding=None, metadata={'file_path': 'D:\\vector_databse\\dataset\\hr_policy_manual.pdf', 'file_name': 'hr_policy_manual.pdf', 'file_type': 'application/pdf', 'file_size': 7583, 'creation_date': '2026-06-13', 'last_modified_date': '2026-06-09'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='d2a758fb-15dc-40cb-ae85-85e59deac25a', node_type='4', metadata={'file_path': 'D:\\vector_databse\\dataset\\hr_policy_manual.pdf', 'file_name': 'hr_policy_manual.pdf', 'file_type': 'application/pdf', 'file_size': 7583, 'creation_date': '2026-06-13', 'last_modified_date': '2026-06-09'}, hash='bd66e4cee8d5b75547eca6cec4e781750ed59c8cc61c2dc55532816

***Metadata filtering**

In [9]:
from llama_index.core.vector_stores.types import MetadataFilters, MetadataFilter

hr_filter = MetadataFilters(
    filters=[
        MetadataFilter(key="file_name", value="hr_policy_manual.pdf")
    ]
)

retriever = index.as_retriever(
    similarity_top_k=3,
    filters=hr_filter
)


query = "What is leave policy?"


In [10]:
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.retrievers import QueryFusionRetriever

In [11]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_vector_store(vs)
retriever = index.as_retriever(similarity_top_k=5)
query_engine = RetrieverQueryEngine.from_args(retriever=retriever)

In [ ]:
from dotenv import load_dotenv; load_dotenv()

True

In [17]:
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

Settings.llm = Ollama(model="llama3.2", request_timeout=120.0)
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [18]:
query_engine = index.as_query_engine()
response = query_engine.query("What is the HR Policy Manual?")
print(response)

ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download